# Data test for beamtime 20251020

In [1]:
from rsm3d.data_io import RSMDataLoader
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
from rsm3d.data_viz import RSMNapariViewer
spec_file = '/Users/xiaogangyang/data/isr_rsm3d/FeTe_GeTe_101525'
setup_file = './exp_setup.yaml'
tiff_dir  = '/Users/xiaogangyang/data/isr_rsm3d/Oct_22_2025'
# tiff_output = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff_cleaned'  
out_vtr   = '/Users/xiaogangyang/data/isr_rsm3d/rsm_hkl_20251020.vtr'    # output file
scan_list = (36,) 

In [2]:
loader = RSMDataLoader(
    spec_file,
    setup_file,
    tiff_dir,
    selected_scans=scan_list,
    process_hklscan_only=True,
)
setup, UB, df = loader.load()
print(setup)
print(UB)

<ExperimentSetup: distance=0.78105 m, pitch=7.5e-05 m, xcenter=515, ycenter=257, xpixels=1030, ypixels=514, theta=15.3069°, phi=0.0°, dtheta=0.04°, energy=11.47 eV, wavelength=1.080943316767221 Å>
[[-0.12235412  1.643203    0.01602708]
 [-1.64319032 -0.12275695  0.01535254]
 [ 0.02709815 -0.02437019  1.00332041]]


In [6]:
print(df.tth, df.th, df.chi, df.phi)

0     22.030848
1     22.030848
2     22.066086
3     22.066086
4     22.101379
        ...    
92    23.832297
93    23.867646
94    23.903050
95    23.938455
96    23.973803
Name: tth, Length: 97, dtype: float64 0     11.89300
1     11.89300
2     11.91065
3     11.91065
4     11.92825
        ...   
92    12.79365
93    12.81135
94    12.82905
95    12.84680
96    12.86440
Name: th, Length: 97, dtype: float64 0     89.084797
1     89.084797
2     89.084797
3     89.084797
4     89.084797
        ...    
92    89.084805
93    89.084805
94    89.084797
95    89.084805
96    89.084805
Name: chi, Length: 97, dtype: float64 0    -0.000837
1    -0.000837
2    -0.000309
3    -0.000309
4    -0.000802
        ...   
92   -0.000665
93   -0.000665
94   -0.000252
95   -0.000309
96   -0.000848
Name: phi, Length: 97, dtype: float64


In [2]:
loader = RSMDataLoader(
    spec_file,
    setup_file,
    tiff_dir,
    selected_scans=scan_list,

    process_hklscan_only=True,
)

builder = RSMBuilder(loader, 
                    sample_axes = ('z-', 'y+', 'x+'),
                    detector_axes = ('z-',),
                    ub_includes_2pi=True)
Q_samp, hkl, intensity = builder.compute_full()

Initialized QConversion area with:
  Sample Axis: ('z-', 'y+', 'x+')
  Detector Axis: ('z-',)
  Beam Direction: (0, 1, 0)
  Wavelength: 1.080943 Å
  Distance: 0.781050 m
  Pixel Width: 0.000075 m


In [3]:
# Optional cropping
# builder.crop_by_positions(y_bound=(220, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(100, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)

In [4]:
viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

# Import the rsm3d module and set the data input/oupt directories

In [1]:
from rsm3d.data_io import RSMDataLoader
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
from rsm3d.data_viz import RSMNapariViewer
spec_file = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23'
setup_file = './exp_setup.yaml'
tiff_dir  = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23_tiff_1017_tmp'
# tiff_output = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff_cleaned'  
out_vtr   = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr'    # output file
scan_list = (21,)  # any list/tuple of scan numbers

# Load the data and call the RSMBuilder and compute the Q-sample and HKL for the listed frames

In [2]:
loader = RSMDataLoader(
    spec_file,
    setup_file,
    tiff_dir,
    selected_scans=scan_list,
    process_hklscan_only=True,
)

builder = RSMBuilder(loader, ub_includes_2pi=True)
Q_samp, hkl, intensity = builder.compute_full()

Initialized QConversion area with:
  Sample Axis: ['x+', 'y+', 'z-']
  Detector Axis: ['x+']
  Beam Direction: (0, 1, 0)
  Wavelength: 1.080943 Å
  Distance: 0.781050 m
  Pixel Width: 0.000075 m


# Mapping the intensity with the HKL/Q_samp for 3D visualization

In [3]:
# Optional cropping
# builder.crop_by_positions(y_bound=(220, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)

# Call the napari for the 3D visualization of the RSM map

In [ ]:
viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

: 

In [11]:
# Optional cropping
builder.crop_by_positions(y_bound=(240, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)
viz = RSMNapariViewer(
    grid, (xax, yax, zax-0.06558),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [9]:
from rsm3d.data_io import write_rsm_volume_to_vtr, write_rsm_volume_to_vtk
rsm = grid
edges = (xax, yax, zax)
filename = out_vtr
write_rsm_volume_to_vtk(rsm, edges, filename.replace('.vtr', '.vtk'))
write_rsm_volume_to_vtr(rsm, edges, filename, binary=False, compress=True)
